# 02 · Maximising AEP with TopFarm (single objective)

Same site, turbines and wake model as notebook 01. Here the turbines are free to move inside a
2 km × 1 km lease area, subject to a minimum spacing of 4 rotor diameters, and TopFarm's
gradient-based optimiser (SLSQP, with exact gradients from autograd) maximises annual energy production.

Cable length is ignored here on purpose. Notebook 03 adds it as a second objective.

In [ ]:
%pip install -q py_wake==2.6.20 topfarm==2.6.2 pymoo==0.6.2

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from py_wake.examples.data.hornsrev1 import Hornsrev1Site, V80
from py_wake.literature.gaussian_models import Bastankhah_PorteAgel_2014
from py_wake.utils.gradients import autograd
from topfarm import TopFarmProblem
from topfarm.cost_models.py_wake_wrapper import PyWakeAEPCostModelComponent
from topfarm.easy_drivers import EasyScipyOptimizeDriver
from topfarm.constraint_components.spacing import SpacingConstraint
from topfarm.constraint_components.boundary import XYBoundaryConstraint

FIG = Path("figures"); FIG.mkdir(exist_ok=True)

site, wt = Hornsrev1Site(), V80()
wf_model = Bastankhah_PorteAgel_2014(site, wt, k=0.0324555)

LEASE_X, LEASE_Y = 2000, 1000        # lease area (m), shared with notebook 03
MIN_SPACING = 4 * wt.diameter()      # 320 m

In [ ]:
x_init = np.linspace(0, 1600, 4).repeat(2)
y_init = np.tile(np.linspace(0, 400, 2), 4)
aep_init = wf_model(x_init, y_init).aep().sum().values

problem = TopFarmProblem(
    design_vars={"x": x_init, "y": y_init},
    cost_comp=PyWakeAEPCostModelComponent(wf_model, n_wt=len(x_init), grad_method=autograd),
    constraints=[SpacingConstraint(MIN_SPACING),
                 XYBoundaryConstraint([(0, 0), (LEASE_X, LEASE_Y)], "rectangle")],
    driver=EasyScipyOptimizeDriver(optimizer="SLSQP", maxiter=200, tol=1e-10),
    expected_cost=aep_init,          # scales the objective so it starts near -1
)
cost, state, recorder = problem.optimize()

x_opt, y_opt = state["x"], state["y"]
aep_opt = wf_model(x_opt, y_opt).aep().sum().values
print(f"Initial grid AEP: {aep_init:.2f} GWh")
print(f"Optimised AEP:    {aep_opt:.2f} GWh  (+{(aep_opt / aep_init - 1) * 100:.2f} %)")

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot([0, LEASE_X, LEASE_X, 0, 0], [0, 0, LEASE_Y, LEASE_Y, 0], "k--", alpha=0.6, label="Lease area")
plt.scatter(x_init, y_init, color="red", marker="x", s=100, label="Initial 4x2 grid")
plt.scatter(x_opt, y_opt, color="green", s=100, label="Optimised layout")
for i in range(len(x_init)):
    plt.annotate("", xy=(x_opt[i], y_opt[i]), xytext=(x_init[i], y_init[i]),
                 arrowprops=dict(arrowstyle="->", color="gray", alpha=0.6))
plt.title("AEP-maximising layout (8 × V80, SLSQP)")
plt.xlabel("x (m)"); plt.ylabel("y (m)"); plt.axis("equal")
plt.legend(); plt.grid(True, alpha=0.4)
plt.savefig(FIG / "02_topfarm_layout.png", dpi=150, bbox_inches="tight")
plt.show()

**Takeaway:** the optimiser pushes turbines towards the edges of the lease area to spread them out
of each other's wakes. That gain in energy comes with longer distances between turbines, which is
the cost notebook 03 makes visible.